In [3]:
from networkx import is_d_separator
from pgmpy.base import DAG

In [6]:
# Reminder that we construct DAGs by defining edges.
# DAG is a base calss for BayesianNetwork class, so objects here will work on 
# DAG related objects and BayesianNetwork objects! 

dag = DAG([  # life goal: get the right projects and build these so frequently that looking at these pairs forms the image of the DAG in my mind
    ('I', 'U'),
    ('I', 'M'),
    ('M', 'U'),
    ('J', 'V'),
    ('J', 'M'),
    ('M', 'V')
])

print(is_d_separator(dag, {"U"}, {"V"}, {"M"}))

False


In [8]:
print(is_d_separator(dag, {"U"}, {"V"}, {"M", "I", "J"}))

True


In [10]:
print(is_d_separator(dag, {"U"}, {"V"}, {"M", "I"}))

True


In [12]:
print(is_d_separator(dag, {"U"}, {"V"}, {"M", "J"}))

True


In [13]:
dag.get_independencies()

(I ⟂ J)
(J ⟂ U | I, M)
(U ⟂ V | I, M)
(I ⟂ V | M, J)

## Refuting DAG

In [29]:
import polars as pl
survey_url = "https://raw.githubusercontent.com/altdeep/causalML/refs/heads/master/datasets/transportation_survey.csv"
full_data = pl.read_csv(survey_url)

In [34]:
data = full_data[:30]
data

A,S,E,O,R,T
str,str,str,str,str,str
"""adult""","""F""","""high""","""emp""","""small""","""train"""
"""young""","""M""","""high""","""emp""","""big""","""car"""
"""adult""","""M""","""uni""","""emp""","""big""","""other"""
"""old""","""F""","""uni""","""emp""","""big""","""car"""
"""young""","""F""","""uni""","""emp""","""big""","""car"""
…,…,…,…,…,…
"""adult""","""F""","""high""","""emp""","""big""","""car"""
"""young""","""M""","""high""","""emp""","""big""","""car"""
"""young""","""M""","""uni""","""emp""","""big""","""car"""


In [39]:
from pgmpy.estimators.CITests import chi_square
significance = .05

result = chi_square(
    X="E", Y="T", Z={"O", "R"},
    data=data.to_pandas(),
    boolean=False,
    significance_level=significance
)

result

(np.float64(1.1611111111111112), np.float64(0.5595873983053805), 2)

In [43]:
from pprint import pprint
from pgmpy.base import DAG
from pgmpy.independencies import IndependenceAssertion

dag = DAG([
    ("A", "E"),
    ("S", "E"),
    ("E", "O"),
    ("E", "R"),
    ("O", "T"),
    ("R", "T")
])

dseps = dag.get_independencies()
pprint(dseps)

(A ⟂ T | R, O)
(R ⟂ S | E)
(S ⟂ A)
(S ⟂ T | R, O)
(R ⟂ O | E)
(A ⟂ O | E)
(E ⟂ T | R, O)
(S ⟂ O | E)
(R ⟂ A | E)


In [49]:
type(dseps.get_assertions())

list

In [44]:
def test_dsep(dsep):
    test_outputs = []
    for X in list(dsep.get_assertion()[0]):
        for Y in list(dsep.get_assertion()[1]):
            Z = list(dsep.get_assertion()[2])
            test_result = chi_square(
                X=X, Y=Y, Z=Z,
                data=data.to_pandas(),
                boolean=True,
                significance_level=significance
            )

            assertion = IndependenceAssertion(X, Y, Z)
            test_outputs.append((assertion, test_result))

    return test_outputs

In [52]:
results = [test_dsep(dsep) for dsep in dseps.get_assertions()]
results = dict([item for sublist in results for item in sublist])
pprint(results)

{(R ⟂ O | E): np.False_,
 (S ⟂ A): np.True_,
 (S ⟂ O | E): np.True_,
 (A ⟂ T | R, O): np.True_,
 (S ⟂ T | R, O): np.True_,
 (R ⟂ S | E): np.True_,
 (A ⟂ O | E): np.True_,
 (E ⟂ T | R, O): np.True_,
 (R ⟂ A | E): np.True_}


In [55]:
num_pass = sum(results.values())
num_dseps = len(dseps.independencies)
num_fail = num_dseps - num_pass
print(num_fail / num_dseps)

0.1111111111111111
